# GazeForge GNM: full-HD GPU render (Colab)

This renders the three required videos (**Left**, **Right**, **Generate Motion**) at **1920×1080** and the **8 portraits** at **1600×2000**, using the code and textures in the public GitHub repo.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. It takes about 1–2 hours. At the end a ZIP called `GazeForge_HD_outputs.zip` downloads, and a copy is saved to your Google Drive.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO = "https://github.com/YuvarajSYali-developer/GazeForge"
import os
if not os.path.exists("/content/GazeForge"):
    !git clone --depth 1 $REPO /content/GazeForge
%cd /content/GazeForge
!ls

In [ ]:
# Blender 4.2 LTS (the version the project is developed and tested with)
import os, subprocess
if not os.path.exists("/content/blender/blender"):
    for v in ["4.2.23", "4.2.20", "4.2.15", "4.2.12", "4.2.9", "4.2.6", "4.2.3", "4.2.0"]:
        url = f"https://download.blender.org/release/Blender4.2/blender-{v}-linux-x64.tar.xz"
        if subprocess.run(["curl", "-sfIL", url], capture_output=True).returncode == 0:
            print("downloading", url)
            !wget -q $url -O /content/blender.tar.xz
            break
    !mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
!/content/blender/blender --version | head -1
!apt-get -qq install -y libxi6 libxkbcommon0 libsm6 > /dev/null 2>&1 || true

In [ ]:
# Settings (change if you like)
RES_VIDEO = "1920x1080"; SAMPLES_VIDEO = 64; FPS = 24
RES_STILL = "1600x2000"; SAMPLES_STILL = 256
B = "/content/blender/blender"
os.makedirs("/content/out/videos", exist_ok=True); os.makedirs("/content/out/renders", exist_ok=True)

In [ ]:
# 1) Left (texture set 02)
!$B -b GazeForge_GNM.blend --python gnm_eyes.py -- --input Left --texture 02 --res $RES_VIDEO --samples $SAMPLES_VIDEO --fps $FPS --gpu --out /content/out/videos/left.mp4 2>&1 | grep -E "gazeforge|Error|Traceback"


In [ ]:
# 2) Right (texture set 06)
!$B -b GazeForge_GNM.blend --python gnm_eyes.py -- --input Right --texture 06 --res $RES_VIDEO --samples $SAMPLES_VIDEO --fps $FPS --gpu --out /content/out/videos/right.mp4 2>&1 | grep -E "gazeforge|Error|Traceback"


In [ ]:
# 3) Generate Motion (texture set 03)
!$B -b GazeForge_GNM.blend --python gnm_eyes.py -- --input "Generate Motion" --texture 03 --res $RES_VIDEO --samples $SAMPLES_VIDEO --fps $FPS --gpu --out /content/out/videos/generate_motion.mp4 2>&1 | grep -E "gazeforge|Error|Traceback"


In [ ]:
# 4) The 8 portraits, one per texture set
!$B -b --python render_stills.py -- --res $RES_STILL --samples $SAMPLES_STILL --gpu --out /content/out/renders 2>&1 | grep -E "stills|Error|Traceback"


In [ ]:
# 5) Check, zip, save to Drive and download
!for f in /content/out/videos/*.mp4; do echo $f; ffprobe -v error -show_entries format=duration:stream=width,height -of compact $f; done
!ls -la /content/out/renders
!cd /content && rm -f GazeForge_HD_outputs.zip && zip -qr GazeForge_HD_outputs.zip out
try:
    from google.colab import drive; drive.mount("/content/drive")
    !cp /content/GazeForge_HD_outputs.zip /content/drive/MyDrive/
    print("saved to Google Drive: MyDrive/GazeForge_HD_outputs.zip")
except Exception as e:
    print("Drive not mounted:", e)
from google.colab import files
files.download("/content/GazeForge_HD_outputs.zip")